## 1. Create Dataset

In [1]:
import numpy as np

def one_hot(val, size):
    vec = [0] * size
    vec[val] = 1
    return vec
              
# Create dataset 
X = np.empty((1, 4), dtype=int)
number = 1
for context in range(2,4):
    for a1 in range(3):
        
        # If mouse goes to cue it sees Right or Left
        if a1 == 2:
            o2 = context
            
        # If it goes Left  or Right it sees Cheese or Shock
        elif a1 == 1:
            o2 = 3 - context
            
        else:
            o2 = context - 2
            
        for a2 in range(3):
            if a1 != 2: # TRAP
                o3 = o2
            else:
                if a2 == 2:
                    o3 = context
            
                # If it goes Left  or Right it sees Cheese or Shock
                elif a2 == 1:
                    o3 = 3 - context
                    
                else:
                    o3 = context - 2
                    
            # print(f"number{number} a1:{a1} o2:{o2} a2:{a2} o3:{o3}")
            number += 1
            datapoint = np.array([a1, o2, a2, o3])

            X = np.vstack((X, datapoint))

  
X = X[1:] # Remove the first empty row          
print(X.shape) # 18 possible paths and 4 variables (a1, o2, a2, o3)


(18, 4)


## 2. Initialize Tensor Net

In [2]:
from tn4ml.models.mps import MatrixProductState
from tn4ml.metrics import NegLogLikelihood
import numpy as np
import optax
from tn4ml.util import EarlyStopping 
import jax.numpy as jnp

def identity_3d_cubical(shape, backend='numpy'):
    """
    Creates a 3D identity-like tensor where only elements with i == j == k are 1.

    Args:
        shape (tuple): Shape of the 3D tensor (e.g., (3, 3, 3)).
        backend (str): 'numpy' or 'jax'. Defaults to 'numpy'.

    Returns:
        ndarray: 3D identity-like tensor.

    Example:
        >>> identity_3d_cubical((3, 3, 3))
        array([[[1, 0, 0],
                [0, 0, 0],
                [0, 0, 0]],
               [[0, 0, 0],
                [0, 1, 0],
                [0, 0, 0]],
               [[0, 0, 0],
                [0, 0, 0],
                [0, 0, 1]]])
    """
    if backend == 'jax':
        xp = jnp
    else:
        xp = np

    def _identity_rule(i, j, k):
        return xp.where(i == j, 1, 0)

    return xp.fromfunction(
        xp.vectorize(_identity_rule, otypes=[float]),
        shape,
        dtype=float
    )
    
# Initialize MPS
max_bond_dim = 16

A1 = identity_3d_cubical((max_bond_dim, max_bond_dim, 4)) # Should be 3, but in order to pass one Embed function, actions will have a 4th null state.
O2 = identity_3d_cubical((max_bond_dim, max_bond_dim, 4))
A2 = identity_3d_cubical((max_bond_dim, max_bond_dim, 4)) # Idem
O3 = identity_3d_cubical((max_bond_dim, max_bond_dim, 4))
tensors = [A1, O2, A2, O3]

model = MatrixProductState(tensors)
print(model)



d:\Documents\Github\tn4mlTmaze\.venv\Lib\site-packages\tn4ml\models\smpo.py:8: FutureWarning: The module 'quimb.tensor.tensor_1d' is deprecated and will be removed in a future release. Most functionality can be still be accessed directly from 'quimb.tensor' instead. The actual implementations have moved to `quimb.tensor.tn1d.core`.
  from quimb.tensor.tensor_1d import MatrixProductState, TensorNetwork1DOperator, TensorNetwork1DFlat


MatrixProductState([
    Tensor(shape=(16, 16, 4), inds=('_17c7a3AAAAA', '_17c7a3AAAAB', 'k0'), tags=oset(['I0'])),
    Tensor(shape=(16, 16, 4), inds=('_17c7a3AAAAB', '_17c7a3AAAAC', 'k1'), tags=oset(['I1'])),
    Tensor(shape=(16, 16, 4), inds=('_17c7a3AAAAC', '_17c7a3AAAAD', 'k2'), tags=oset(['I2'])),
    Tensor(shape=(16, 16, 4), inds=('_17c7a3AAAAD', '_17c7a3AAAAA', 'k3'), tags=oset(['I3'])),
], tensors=4, indices=8, L=4, max_bond=16)


## 3. Embed data with a one hot feature map

In [3]:


from tn4ml.embeddings import Embedding

class OneHotEmbedding(Embedding):
    """One-hot feature map."""
    def __init__(self, num_states: int, **kwargs):
        assert num_states >= 1
        self.num_states = num_states
        super().__init__(**kwargs)

    @property
    def dim(self) -> int:
        return self.num_states

    @property
    def input_dim(self) -> int:
        return 1

    def __call__(self, x: int) -> jnp.ndarray:
        x_int = np.round(x).astype(int)
        one_hot = jnp.zeros(self.num_states)
        one_hot = one_hot.at[x_int].set(1.0)
        return one_hot

embedding = OneHotEmbedding(num_states=4)



learning_rate = 1e-4
earlystop = EarlyStopping(min_delta=0, patience=10, monitor='loss', mode='min')
device = 'cpu'

# define training parameters
epochs = 100
batch_size = 18
optimizer = optax.adam
strategy = 'global'
loss = NegLogLikelihood
train_type = 0 # TrainingType.UNSUPERVISED

model.configure(optimizer=optimizer, strategy=strategy, loss=loss, train_type=train_type, learning_rate=learning_rate, device=device)

## 4. Train TN

In [4]:

history = model.train(X,
            epochs=epochs,
            batch_size=batch_size,
            embedding = embedding,
            normalize = True,
            dtype = jnp.float64,
            earlystop = earlystop,
            cache = True,
            )

d:\Documents\Github\tn4mlTmaze\.venv\Lib\site-packages\tn4ml\models\model.py:303: UserWarning: Explicitly requested dtype float64 requested in ones is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  dummy_input = jnp.ones(shape=input_shape, dtype=inputs_dtype)


Tensor(shape=(1, 4), inds=('_17c7a3AAAAF', 'k0'), tags=oset(['I0']), backend='jax', dtype='float32')
Tensor(shape=(1, 4), inds=('_17c7a3AAAAJ', 'k0'), tags=oset(['I0']), backend='jax', dtype='float32')


epoch:   0%|          | 0/100 [00:00<?, ?it/s]d:\Documents\Github\tn4mlTmaze\.venv\Lib\site-packages\tn4ml\models\model.py:946: UserWarning: Explicitly requested dtype float64 requested in asarray is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  x = jax.numpy.asarray(x, dtype=dtype)
epoch:   1%|           1/100 , loss=-5.5452   

Waiting for 1 epochs.


epoch:   1%|           1/100 , loss=5.5451 

...

epoch:   8%|▊          8/100 , loss=5.5424

Waiting for 9 epochs.


epoch:   9%|▉          9/100 , loss=5.5420

Training stopped by EarlyStopping on epoch: 0


epoch:  11%|█          11/100 , loss=5.5416


## 4. Evaluate TN

In [5]:
import utils as u
import tensornetwork as tn

def tensorNet(mps):
    # Convert MPS to TensorNetwork
    nodes = []
    conj_nodes = []
    for idx, tensor in enumerate(mps):
        nodes.append(tn.Node(tensor, name=f"node_{idx}"))
        conj_nodes.append(tn.Node(np.conj(tensor), name=f"conj_node_{idx}"))

    return nodes, conj_nodes

def von_neumann_entropy(rho, base=2):
    """Computes the von Neumann entropy of a density matrix."""
    # Filter out zero eigenvalues to avoid log(0)
    eigenvalues = np.linalg.eigvalsh(rho)
    non_zero_eigenvalues = eigenvalues[eigenvalues > 1e-12]
    return u.shannon_entropy(non_zero_eigenvalues, base=base)

def get_rdm(nodes, conj_nodes, sites):
    """
    Computes the reduced density matrix (RDM) for a given set of (ascending) sites.
    """

    print("\n \n Initial sites:", sites)

    # Due to left-canonicalization, we only need to contract up to the rightmost site.
    max_site = max(sites)
    nodes = nodes[:max_site + 1]
    conj_nodes = conj_nodes[:max_site + 1]
    #print("\n Nodes1:", nodes)

    # 1. Connect uninformative edges
    nodes[0][0] ^ conj_nodes[0][0]
    nodes[-1][2] ^ conj_nodes[-1][2]

    # 2. Connect the virtual "bond" indices of the MPS chain
    for i in range(max_site):
        nodes[i][2] ^ nodes[i+1][0]
        conj_nodes[i][2] ^ conj_nodes[i+1][0]

    # Define the open edges for the RDM
    open_edges = []
    for site_idx in sites:
        open_edges.append(nodes[site_idx][1])
    for site_idx in sites:
        open_edges.append(conj_nodes[site_idx][1])

    # Create a list of sites to be contracted (traced out)
    sites_to_contract = set(range(max_site + 1)) - sites

    # 3. Connect the physical indices of the sites to be traced out
    for i in sites_to_contract:
        nodes[i][1] ^ conj_nodes[i][1]

    # Contract the network to get the RDM tensor
    rdm_node = tn.contractors.auto(nodes + conj_nodes, output_edge_order=open_edges)
    rdm_tensor = rdm_node.tensor

    # Multiply the first half of the shape vector to get the matrix dimensions
    rdm_rows = int(np.prod(rdm_tensor.shape[:len(rdm_tensor.shape)//2]))
    print("\n RDM tensor shape:", rdm_tensor.shape)

    # Reshape into a matrix
    rdm = rdm_tensor.reshape(rdm_rows, rdm_rows)

    return rdm


ModuleNotFoundError: No module named 'torch'

In [ ]:
mps = []
for tensor in model.tensors:
    mps.append(tensor.data)
    
nodes, conj_nodes = tensorNet(mps)
rho_a1s2 = get_rdm(nodes, conj_nodes, sites=[0, 1])
rho_a1s2 = rho_a1s2 / np.trace(rho_a1s2)

p_a1s2 = np.real(np.diag(rho_a1s2)).reshape(4, -1)

# Obtain conditional distribution p(s2|a1 ) by normalizing over s2 for each a1
p_s2_given_a1 = p_a1s2 / np.sum(p_a1s2, axis=1, keepdims=True)
u.plot_distribution(p_s2_given_a1, title="p( s2 | a1 )")


(16, 16, 4)
